In [ ]:
#!/usr/bin/env python3
"""
Cell-type-specific Mendelian Randomization (csMR) Pipeline
============================================================
Exposure: GWAS summary statistics
Outcome:  SingleBrain sn-eQTL per cell subtype
IVs:      Genome-wide significant SNPs (P<5e-8, LD clumped)
Methods:  IVW, MR-Egger, Weighted Median, Steiger, Cochran's Q
"""

import os, time, gzip, subprocess, tempfile
import numpy as np, pandas as pd
from scipy import stats
from concurrent.futures import ProcessPoolExecutor, as_completed


# ========================= Configuration =========================

# --- Exposure GWAS ---
EXPOSURE_GWAS = "./data/gwas/your_gwas_summary_stats.txt"          # Path to GWAS summary statistics
EXPOSURE_NAME = "demo_trait"          # Short trait name (used in output filenames)
EXPOSURE_SEP  = "\t"        # Column separator

# Column name mapping for exposure GWAS
EXPOSURE_COL_SNP  = "SNP"
EXPOSURE_COL_CHR  = "chr"
EXPOSURE_COL_POS  = "pos"
EXPOSURE_COL_BETA = "beta"
EXPOSURE_COL_SE   = "se"
EXPOSURE_COL_PVAL = "pval"
EXPOSURE_COL_EA   = "effect_allele"
EXPOSURE_COL_OA   = "other_allele"
EXPOSURE_COL_EAF  = "eaf"
EXPOSURE_COL_N    = "N"

# --- SingleBrain eQTL ---
SINGLEBRAIN_DIR = "./data/visium/"        # Directory containing {celltype}_eqtl_full_assoc.tsv.gz
N_EQTL = 983               # eQTL sample size

# --- Causal gene list (optional, for filtering) ---
CAUSAL_GENES_FILES = []     # List of CSV paths from gsMap causal gene analysis
GSMAP_FDR = 0.05

# --- LD clumping ---
PLINK  = "./data/tools/plink/"                 # Path to plink binary
LD_REF = "./data/ld_reference/1000G.EUR.QC"                 # LD reference panel prefix (e.g. 1000G.EUR.QC)
CLUMP_P1 = 5e-8
CLUMP_R2 = 0.001
CLUMP_KB = 10000

# --- GTF annotation (for gene name mapping) ---
GTF_FILE = "./data/annotation/gencode.vsj.annotation.gtf"               # Path to gencode GTF file

# --- MR parameters ---
F_STAT_MIN  = 10
NUM_WORKERS = 80

# --- Output ---
OUTPUT_DIR = "./results/"             # Output directory


# ========================= Cell Types =========================

SB_CELLTYPES = [
    "Ext", "Ext1", "Ext2", "Ext3", "Ext4", "Ext5", "Ext6", "Ext7", "Ext8",
    "IN", "IN1", "IN2", "IN3", "IN4", "IN5", "IN6", "IN7",
    "Ast", "Ast1", "Ast2", "Ast3", "Ast4",
    "MG", "MG1", "MG2", "MG3", "MG4", "MiGA3",
    "OD", "OD1", "OD2", "OD3",
    "OPC", "OPC1", "OPC2",
    "End",
]


# ========================= Utilities =========================

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")


# ========================= MR Methods =========================

def mr_ivw(bx, by, sx, sy):
    """Inverse-variance weighted MR."""
    w = 1.0 / sy**2
    beta = np.sum(w * bx * by) / np.sum(w * bx**2)
    se = np.sqrt(1.0 / np.sum(w * bx**2))
    z = beta / se
    p = 2 * stats.norm.sf(abs(z))
    return beta, se, p


def mr_egger(bx, by, sx, sy):
    """MR-Egger regression. Returns (slope, slope_se, slope_p, intercept, intercept_p)."""
    w = 1.0 / sy**2
    n = len(bx)
    if n < 3:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    X = np.column_stack([np.ones(n), bx])
    W = np.diag(w)
    try:
        coef = np.linalg.solve(X.T @ W @ X, X.T @ W @ by)
        resid = by - X @ coef
        s2 = np.sum(w * resid**2) / (n - 2)
        cov = s2 * np.linalg.inv(X.T @ W @ X)
        intercept = coef[0]; intercept_se = np.sqrt(cov[0, 0])
        slope = coef[1]; slope_se = np.sqrt(cov[1, 1])
        intercept_p = 2 * stats.norm.sf(abs(intercept / intercept_se))
        slope_p = 2 * stats.norm.sf(abs(slope / slope_se))
        return slope, slope_se, slope_p, intercept, intercept_p
    except Exception:
        return np.nan, np.nan, np.nan, np.nan, np.nan


def mr_weighted_median(bx, by, sx, sy, n_boot=1000):
    """Weighted median MR with bootstrap SE."""
    ratio = by / bx
    w = 1.0 / (sy**2 / bx**2)
    w = w / w.sum()
    sort_idx = np.argsort(ratio)
    ratio_s = ratio[sort_idx]; w_s = w[sort_idx]
    cumw = np.cumsum(w_s)
    idx = np.searchsorted(cumw, 0.5)
    beta = ratio_s[min(idx, len(ratio_s) - 1)]

    np.random.seed(42)
    betas = []
    for _ in range(n_boot):
        by_b = by + np.random.normal(0, sy)
        bx_b = bx + np.random.normal(0, sx)
        valid = bx_b != 0
        if valid.sum() < 3:
            continue
        r = by_b[valid] / bx_b[valid]
        wb = 1.0 / (sy[valid]**2 / bx_b[valid]**2)
        wb = wb / wb.sum()
        si = np.argsort(r); rs = r[si]; ws = wb[si]
        cw = np.cumsum(ws)
        ii = np.searchsorted(cw, 0.5)
        betas.append(rs[min(ii, len(rs) - 1)])
    se = np.std(betas) if betas else np.nan
    p = 2 * stats.norm.sf(abs(beta / se)) if se > 0 else np.nan
    return beta, se, p


def cochrans_q(bx, by, sy, ivw_beta):
    """Cochran's Q test for heterogeneity."""
    w = 1.0 / sy**2
    q = np.sum(w * (by - ivw_beta * bx)**2)
    df = len(bx) - 1
    p = 1 - stats.chi2.cdf(q, df) if df > 0 else np.nan
    return q, p


def steiger_test(bx, by, sx, sy, n_exp, n_out):
    """Steiger directionality test."""
    r2_exp = (bx**2) / (bx**2 + n_exp * sx**2)
    r2_out = (by**2) / (by**2 + n_out * sy**2)
    mean_r2_exp = np.mean(r2_exp)
    mean_r2_out = np.mean(r2_out)
    correct = mean_r2_exp > mean_r2_out
    return mean_r2_exp, mean_r2_out, correct


# ========================= Per-Gene MR Worker =========================

def do_mr_one(args):
    """Run MR for one gene x cell type combination."""
    gene, ensg, ct, iv_snps, eqtl_cache_dir, n_exp = args
    try:
        cache_file = os.path.join(eqtl_cache_dir, ct, f"{gene}.csv.gz")
        if not os.path.exists(cache_file):
            return None
        eqtl = pd.read_csv(cache_file)

        merged = iv_snps.merge(eqtl, on="SNP", how="inner", suffixes=("_exp", "_out"))
        if len(merged) < 3:
            return None

        bx = merged["beta_exp"].values
        by = merged["BETA"].values
        sx = merged["se_exp"].values
        sy = merged["SE"].values

        valid = (sx > 0) & (sy > 0) & np.isfinite(bx) & np.isfinite(by)
        if valid.sum() < 3:
            return None
        bx = bx[valid]; by = by[valid]; sx = sx[valid]; sy = sy[valid]

        F = (bx / sx) ** 2
        if np.mean(F) < F_STAT_MIN:
            return None

        ivw_beta, ivw_se, ivw_p = mr_ivw(bx, by, sx, sy)
        egger_beta, egger_se, egger_p, egger_int, egger_int_p = mr_egger(bx, by, sx, sy)
        wm_beta, wm_se, wm_p = mr_weighted_median(bx, by, sx, sy)
        q_stat, q_p = cochrans_q(bx, by, sy, ivw_beta)

        n_out = N_EQTL
        s_r2_exp, s_r2_out, s_correct = steiger_test(bx, by, sx, sy, n_exp, n_out)

        direction = "up" if ivw_beta > 0 else "down"

        return {
            "gene": gene, "ensg": ensg, "cell_type": ct,
            "n_iv": len(bx), "mean_F": round(np.mean(F), 1),
            "ivw_beta": round(ivw_beta, 4), "ivw_se": round(ivw_se, 4),
            "ivw_pvalue": ivw_p, "ivw_method": "IVW",
            "direction": direction,
            "steiger_r2_exp": round(s_r2_exp, 6), "steiger_r2_out": round(s_r2_out, 6),
            "steiger_correct": s_correct,
            "q_stat": round(q_stat, 2), "q_pvalue": q_p,
            "egger_beta": round(egger_beta, 4) if not np.isnan(egger_beta) else np.nan,
            "egger_se": round(egger_se, 4) if not np.isnan(egger_se) else np.nan,
            "egger_pvalue": egger_p,
            "egger_intercept": round(egger_int, 6) if not np.isnan(egger_int) else np.nan,
            "egger_intercept_p": egger_int_p,
            "wm_beta": round(wm_beta, 4) if not np.isnan(wm_beta) else np.nan,
            "wm_se": round(wm_se, 4) if not np.isnan(wm_se) else np.nan,
            "wm_pvalue": wm_p,
        }
    except Exception:
        return None


# ========================= Main =========================

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    log("=" * 70)
    log(f"csMR Pipeline: {EXPOSURE_NAME} -> gene expression (SingleBrain)")
    log("=" * 70)

    # --- [1] Load exposure GWAS ---
    log("\n[1] Loading exposure GWAS...")
    gwas = pd.read_csv(EXPOSURE_GWAS, sep=EXPOSURE_SEP)
    gwas = gwas.rename(columns={
        EXPOSURE_COL_SNP: "SNP", EXPOSURE_COL_CHR: "chr", EXPOSURE_COL_POS: "pos",
        EXPOSURE_COL_BETA: "beta", EXPOSURE_COL_SE: "se", EXPOSURE_COL_PVAL: "pval",
        EXPOSURE_COL_EA: "effect_allele", EXPOSURE_COL_EAF: "eaf", EXPOSURE_COL_N: "samplesize",
    })
    gwas = gwas.dropna(subset=["SNP", "beta", "se", "pval"])
    gwas = gwas[gwas["se"] > 0]
    n_gwas = int(gwas["samplesize"].median())
    log(f"  {len(gwas)} SNPs, N={n_gwas}")

    # --- [2] Genome-wide significant SNPs ---
    sig = gwas[gwas["pval"] < CLUMP_P1]
    log(f"\n[2] P < {CLUMP_P1}: {len(sig)} SNPs")

    # --- [3] LD clumping ---
    log("\n[3] LD clumping...")
    iv_file = os.path.join(OUTPUT_DIR, f"{EXPOSURE_NAME}_clumped_IVs.csv")
    if os.path.exists(iv_file):
        ivs = pd.read_csv(iv_file)
        log(f"  Loaded existing: {len(ivs)} IVs")
    else:
        with tempfile.TemporaryDirectory() as tmp:
            sig_file = os.path.join(tmp, "sig.txt")
            sig[["SNP", "pval"]].to_csv(sig_file, sep="\t", index=False)

            all_ivs = []
            for chrom in range(1, 23):
                bfile = f"{LD_REF}.{chrom}"
                if not os.path.exists(f"{bfile}.bim"):
                    continue
                clump_out = os.path.join(tmp, f"clump_{chrom}")
                cmd = (f"{PLINK} --bfile {bfile} --clump {sig_file} "
                       f"--clump-p1 {CLUMP_P1} --clump-r2 {CLUMP_R2} --clump-kb {CLUMP_KB} "
                       f"--out {clump_out} --silent")
                subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=300)
                clumped_file = clump_out + ".clumped"
                if os.path.exists(clumped_file):
                    cl = pd.read_csv(clumped_file, sep=r"\s+")
                    if "SNP" in cl.columns:
                        all_ivs.extend(cl["SNP"].tolist())

            ivs = gwas[gwas["SNP"].isin(all_ivs)].copy()
            ivs["F_stat"] = (ivs["beta"] / ivs["se"]) ** 2
            ivs = ivs[ivs["F_stat"] >= F_STAT_MIN]
            ivs.to_csv(iv_file, index=False)

        log(f"  Clumped: {len(ivs)} IVs (F >= {F_STAT_MIN})")

    log(f"  F-statistic: min={ivs['F_stat'].min():.0f}, "
        f"median={ivs['F_stat'].median():.0f}, max={ivs['F_stat'].max():.0f}")

    # --- [4] Extract IV SNP effects from eQTL ---
    log("\n[4] Extracting eQTL cache...")
    eqtl_cache = os.path.join(OUTPUT_DIR, "eqtl_iv_cache")

    # Collect causal genes (optional filter)
    causal_genes = set()
    for cf in CAUSAL_GENES_FILES:
        if os.path.exists(cf):
            cg = pd.read_csv(cf)
            sig_cg = cg[(cg["correlation"] > 0) & (cg["pvalue"] < 0.05 / len(cg))]
            causal_genes.update(sig_cg["gene"].tolist())
            log(f"  {os.path.basename(cf)}: {len(sig_cg)} significant genes")
    log(f"  Total causal genes: {len(causal_genes)}")

    iv_snps_set = set(ivs["SNP"])
    iv_df = ivs[["SNP", "beta", "se", "pval", "eaf"]].copy()
    iv_df.columns = ["SNP", "beta_exp", "se_exp", "pval_exp", "eaf_exp"]

    # Extract per-celltype IV SNP effects
    n_cached = 0; n_todo = 0
    for ct in SB_CELLTYPES:
        ct_cache = os.path.join(eqtl_cache, ct)
        os.makedirs(ct_cache, exist_ok=True)

        eqtl_file = os.path.join(SINGLEBRAIN_DIR, f"{ct}_eqtl_full_assoc.tsv.gz")
        if not os.path.exists(eqtl_file):
            continue

        done_flag = os.path.join(ct_cache, "_DONE")
        if os.path.exists(done_flag):
            n_cached += 1; continue
        n_todo += 1

        log(f"  Extracting {ct}...")
        gene_data = {}
        with gzip.open(eqtl_file, "rt") as f:
            header = f.readline().strip().split("\t")
            i_f = header.index("feature")
            i_v = header.index("variant_id")
            i_b = header.index("fixed_beta")
            i_s = header.index("fixed_sd")
            i_p = header.index("Fixed_P")

            for line in f:
                parts = line.strip().split("\t")
                snp = parts[i_v]
                if snp not in iv_snps_set:
                    continue

                gene_full = parts[i_f]
                gene_base = gene_full.split(".")[0]

                try:
                    beta = float(parts[i_b])
                    se = float(parts[i_s])
                    p = float(parts[i_p])
                except Exception:
                    continue

                if gene_base not in gene_data:
                    gene_data[gene_base] = []
                gene_data[gene_base].append({"SNP": snp, "BETA": beta, "SE": se, "P": p})

        n_saved = 0
        for gene_base, rows in gene_data.items():
            df = pd.DataFrame(rows).drop_duplicates(subset="SNP", keep="first")
            if len(df) >= 3:
                df.to_csv(os.path.join(ct_cache, f"{gene_base}.csv.gz"),
                          index=False, compression="gzip")
                n_saved += 1

        with open(done_flag, "w") as f:
            f.write(f"genes={n_saved}\n")
        log(f"    {n_saved} genes with >= 3 IVs")

    log(f"  Cached: {n_cached}, newly extracted: {n_todo}")

    # Build ENSG -> gene name mapping
    log("\n  Building gene name mapping...")
    gene_coords_file = os.path.join(OUTPUT_DIR, "gene_coords.csv")
    if os.path.exists(gene_coords_file):
        gdf = pd.read_csv(gene_coords_file)
    else:
        cmd = f"grep -P '\\tgene\\t' {GTF_FILE}"
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        recs = []
        for l in r.stdout.strip().split("\n"):
            p = l.split("\t")
            if len(p) < 9:
                continue
            ch = p[0].replace("chr", ""); st = int(p[3]); en = int(p[4]); strand = p[6]
            gid = gn = ""
            for fi in p[8].split(";"):
                fi = fi.strip()
                if fi.startswith("gene_id"):
                    gid = fi.split('"')[1] if '"' in fi else ""
                elif fi.startswith("gene_name"):
                    gn = fi.split('"')[1] if '"' in fi else ""
            if gid and gn:
                tss = st if strand == "+" else en
                recs.append({"ensg": gid, "ensg_base": gid.split(".")[0],
                             "symbol": gn, "chr": ch, "tss": tss})
        gdf = pd.DataFrame(recs).drop_duplicates(subset="symbol", keep="first")
        gdf.to_csv(gene_coords_file, index=False)

    ensg_to_name = dict(zip(gdf["ensg_base"], gdf["symbol"]))
    log(f"  {len(ensg_to_name)} gene mappings")

    # --- [5] Run MR ---
    log("\n[5] Running MR...")
    tasks = []
    for ct in SB_CELLTYPES:
        ct_cache = os.path.join(eqtl_cache, ct)
        if not os.path.exists(ct_cache):
            continue

        for f in os.listdir(ct_cache):
            if not f.endswith(".csv.gz") or f.startswith("_"):
                continue
            ensg = f.replace(".csv.gz", "")
            gene_name = ensg_to_name.get(ensg, ensg)

            # Filter to causal genes if list is provided
            if causal_genes and gene_name not in causal_genes:
                continue

            tasks.append((gene_name, ensg, ct, iv_df, eqtl_cache, n_gwas))

    log(f"  Tasks: {len(tasks)} (gene x cell subtype)")

    results = []; done = 0; t0 = time.time()
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as exe:
        futs = {exe.submit(do_mr_one, t): t for t in tasks}
        for fut in as_completed(futs):
            r = fut.result(); done += 1
            if r:
                results.append(r)
            if done % 5000 == 0:
                log(f"  Progress: {done}/{len(tasks)} "
                    f"({(time.time() - t0) / 60:.1f} min, valid: {len(results)})")

    elapsed = (time.time() - t0) / 60
    log(f"  Completed: {elapsed:.1f} min, valid results: {len(results)}")

    if not results:
        log("  No valid results")
        return

    # --- [6] FDR correction + filtering ---
    log("\n[6] FDR correction...")
    df = pd.DataFrame(results)
    df["ivw_fdr"] = stats.false_discovery_control(df["ivw_pvalue"].values, method="bh")
    df = df.sort_values("ivw_pvalue")

    # Save all results
    df.to_csv(os.path.join(OUTPUT_DIR, f"csMR_{EXPOSURE_NAME}_all.csv"), index=False)

    log(f"  Total tests: {len(df)}")
    log(f"  FDR < 0.05: {(df['ivw_fdr'] < 0.05).sum()} "
        f"({df[df['ivw_fdr'] < 0.05]['gene'].nunique()} genes)")

    # Strict filtering: FDR + no heterogeneity + no pleiotropy
    strict = df[
        (df["ivw_fdr"] < 0.05) &
        ((df["q_pvalue"] > 0.05) | df["q_pvalue"].isna()) &
        ((df["egger_intercept_p"] > 0.05) | df["egger_intercept_p"].isna())
    ]
    log(f"  Strict (FDR<0.05 + Q>0.05 + Egger>0.05): "
        f"{len(strict)} ({strict['gene'].nunique()} genes)")
    strict.to_csv(os.path.join(OUTPUT_DIR, "strict_for_coloc.csv"), index=False)

    # Direction summary
    if len(strict) > 0:
        log(f"\n  Direction: up={strict['direction'].eq('up').sum()}, "
            f"down={strict['direction'].eq('down').sum()}")

    # Per cell class summary
    def get_class(ct):
        for c in ["Ext", "IN", "Ast", "MG", "OD", "OPC", "End", "MiGA"]:
            if ct.startswith(c):
                return c
        return "Other"

    if len(strict) > 0:
        strict_c = strict.copy()
        strict_c["sb_class"] = strict_c["cell_type"].apply(get_class)
        log("\n  Per cell class:")
        for cls in sorted(strict_c["sb_class"].unique()):
            cs = strict_c[strict_c["sb_class"] == cls]
            log(f"    {cls:5s}: {cs['gene'].nunique():4d} genes, {len(cs):4d} pairs")

    # Volcano plot data
    volcano_dir = os.path.join(OUTPUT_DIR, "volcano_data")
    os.makedirs(volcano_dir, exist_ok=True)
    for ct in df["cell_type"].unique():
        ct_df = df[df["cell_type"] == ct][["gene", "ivw_beta", "ivw_pvalue", "ivw_fdr", "direction"]]
        ct_df.to_csv(os.path.join(volcano_dir, f"{ct}.csv"), index=False)

    log(f"\nAll results saved to: {OUTPUT_DIR}")
    log(f"  csMR_{EXPOSURE_NAME}_all.csv: {len(df)} rows")
    log(f"  strict_for_coloc.csv: {len(strict)} rows")
    log(f"  volcano_data/: {df['cell_type'].nunique()} cell subtypes")
    log("Pipeline completed.")


if __name__ == "__main__":
    main()